### Setting the topology

In a YAML file, define the parameters and topology for your test in a similar manner as the following:
```yaml
topology:
  name: "fcquic_relay_eval_multisite"
  wall_time: "2hr"
  relay_nodes: true # whether to add one relay machine in each cluster
  netns_per_client: 5 # number of network namespaces to run on each client
  # see the possible frrouting version at https://deb.frrouting.org/
  frrouting_version: "frr-10.4"
  router_template: "base_router_config_ospf.frr" # path to the router configuration template

  server:
    cluster: "chirop" # Lille
    nodes: 1
    # node: "chirop-5.lille.grid5000.fr"   # optional: pin a specific machine

  # each site has one router + num_clients clients and one relay,
  # all reserved in the given cluster. `name` is used to build role names:
  #   router_<name>, client_<name>, relay_<name>
  sites:
    - name: nancy
      cluster: gros
      num_clients: 5
    - name: rennes
      cluster: parasilo
      num_clients: 5
    - name: nantes
      cluster: ecotype
      num_clients: 5
    - name: lyon
      cluster: nova
      num_clients: 5

  # links are established between routers in different clusters
  # GRE tunnels are established between the two routers, with OSPF running over it.
  # endpoints must be router_server or router_<site name>.
  links:
    - [router_server, router_client_0]             # src -> nancy
    - [router_client_0, router_client_1]           # nancy -> rennes
    - [router_client_0, router_client_3]           # nancy -> lyon
    - [router_client_1, router_client_2]           # rennes -> nantes
``` 


In [ ]:
!pip install enoslib ipywidgets==8.1.5 fabric --break-system-packages


### Setting up the experiment
Once you have your `topology.yaml` file, you can create the `G5KExpe` class which will handle most things for you.

In [1]:
from g5k_eval import G5KExpe

experiment = G5KExpe(
    # change the path to point to your topology yaml file
    topology_conf="./relays.yaml",
    #
    # other parameters exist:
    # g5k_conf_file_loc points to your .python-grid5000.yaml file which contains your grid5000 credentials, by default it is in `~/` (so `/home/USERNAME`)
    # g5k_conf_file_loc=".python-grid5000.yaml"
    #
    # job_type should be deploy, but you may need it to be different
    # job_type="deploy"
    #
    # os_env_name defines the OS environement that is deployed on the machines
    # by default it is debian12 with NFS, however you can find the entire list at https://www.grid5000.fr/w/Getting_Started#:~:text=On%20Grid%275000%20reference%20environments
    # Make sure to pick debian to ensure that the packages are properly installed
    # os_env_name="debian12-nfs"
)

# you should always follow grid5000's usage policy (see https://www.grid5000.fr/w/Grid5000:UsagePolicy)
# this method simply checks that the job you are trying to start will not cross the day-night boundary.
# If it does, it'll warn you. You can always comment this out if you wish...
experiment.usage_policy_check()

provider = experiment.setup_enoslib_conf()

[WARNING]: failed to patch stdout/stderr for fork-safety: 'OutStream' object
has no attribute 'buffer'
[WARNING]: failed to reconfigure stdout/stderr with custom encoding error
handler: 'OutStream' object has no attribute 'reconfigure'


_____        ___  ____  _ _ _
 | ____|_ __  / _ \/ ___|| (_) |__
 |  _| | '_ \| | | \___ \| | | '_ \
 | |___| | | | |_| |___) | | | |_) |
 |_____|_| |_|\___/|____/|_|_|_.__/  10.9.0

 • Documentation: ]8;id=810813;https://discovery.gitlabpages.inria.fr/enoslib/\https://discovery.gitlabpages.inria.fr/enoslib/]8;;\                            
 • Source: ]8;id=733953;https://gitlab.inria.fr/discovery/enoslib\https://gitlab.inria.fr/discovery/enoslib]8;;\                                         
 • Chat: ]8;id=796245;https://framateam.org/enoslib\https://framateam.org/enoslib]8;;\

                         Dependency check                         
┏━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider      ┃    Status     ┃ Hint                           ┃
┡━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Chameleon     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonKVM  │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ ChameleonEdge │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Fabric        │ NOT INSTALLED │ pip install enoslib[fabric]    │
│ Distem        │ NOT INSTALLED │ pip install enoslib[distem]    │
│ IOT-lab       │ NOT INSTALLED │ pip install enoslib[iotlab]    │
│ Grid'5000     │   INSTALLED   │                                │
│ Openstack     │ NOT INSTALLED │ pip install enoslib[chameleon] │
│ Vagrant       │ NOT INSTALLED │ pip install enoslib[vagrant]   │
│ VMonG5k       │   INSTALLED   │                                │
└───────────────┴───────────────┴────────────────────────────────┘

                                Connectivity check                                 
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider  ┃ Key                 ┃ Connectivity ┃ Hint                           ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Grid'5000 │ ssh:access          │      ✅      │ Connection to access.grid5000… │
│ Grid'5000 │ ssh:access:frontend │      ✅      │ Connection Host(rennes.grid50… │
│ Grid'5000 │ api:access          │      ✅      │                                │
│ VMonG5k   │ access              │      ❔      │ Check G5k status               │
└───────────┴─────────────────────┴──────────────┴────────────────────────────────┘


### Reserving resources
Now that G5K is setup, we can create the experiment's reservation by defining the number of machines of each role and in each cluster.

Once done, we proceed with the actual reservation of the machines. Be aware that this step may take some time (minimum 5 minutes). This is due to the deployment of the VM image. 

Don't forget to run "ssh-add KEY_PATH" to allow ansible to connect using your ssh key

In [2]:
experiment.reserve_res(provider)

Reserving resources now, might take a while...


INFO     [ProviderS] Common reservation_date=2026-09-14T14:26:56 (local time) ]8;id=139276;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py\providers.py]8;;\:]8;id=746846;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/providers.py#60\60]8;;\
         [2 providers]                                                                       

INFO     [G5k] Submitting {'name': 'fcquic_relay_eval_multisite',        ]8;id=974059;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=302268;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'types': ['deploy', 'origin=enoslib_g5k'], 'resources': "{clust                     
         er='gros'}/nodes=1+{cluster='gros'}/nodes=1+{cluster='gros'}/no                     
         des=1+slash_22=1,walltime=2:00:00", 'command': 'sleep                               
         31536000', 'queue': 'default', 'reservation': '2026-09-14                           
         14:26:57'} on nancy                                                                 

INFO     [G5k] Submitting {'name': 'fcquic_relay_eval_multisite',        ]8;id=489481;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=59763;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#306\306]8;;\
         'types': ['deploy', 'origin=enoslib_g5k'], 'resources': "{clust                     
         er='parasilo'}/nodes=1+{cluster='parasilo'}/nodes=1+slash_22=1,                     
         walltime=2:00:00", 'command': 'sleep 31536000', 'queue':                            
         'default', 'reservation': '2026-09-14 14:28:23'} on rennes                          

INFO     [G5k] Reloading 6926408 from nancy                              ]8;id=275644;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=213149;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

...

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Results : []


Finished 5 tasks (Gather facts,Ensure apt keyring directory exists,Download FRR GPG key,Add 
FRR apt repository,Install FRR packages) on {'parasilo-18.rennes.grid5000.fr', 
'gros-82.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

#### Setting up interfaces, IP subnets, and Network namespaces

In [3]:
experiment.setup_interfaces()
experiment.assign_node_ips()
experiment.netns_setup_macvlan()

Output()

Prod interface for gros-82.nancy.grid5000.fr: eno1
Prod interface for gros-83.nancy.grid5000.fr: eno1
Prod interface for gros-87.nancy.grid5000.fr: eno1
Prod interface for parasilo-18.rennes.grid5000.fr: eno1
Prod interface for parasilo-26.rennes.grid5000.fr: eno1
Production interfaces per node: {'gros-82.nancy.grid5000.fr': 'eno1', 'gros-83.nancy.grid5000.fr': 'eno1', 'gros-87.nancy.grid5000.fr': 'eno1', 'parasilo-18.rennes.grid5000.fr': 'eno1', 'parasilo-26.rennes.grid5000.fr': 'eno1'}
Mapping of subnet to clusters: {'gros': '10.144.12.0'}
Adding ip 10.158.8.1 to host: parasilo-18.rennes.grid5000.fr
Adding ip 10.158.8.2 to host: parasilo-26.rennes.grid5000.fr


Finished 1 tasks (cmd) on {'parasilo-26.rennes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Node IPs: {'router_server': ['10.158.8.1', '172.16.97.18'], 'server': ['10.158.8.2'], 'router_client_0': ['10.144.12.1', '172.16.66.82'], 'client_0': ['10.144.12.2'], 'relay_0': ['10.144.12.3']}
number of total ip addresses (i.e., indiviual client): 1
All Network namespaces IPs: ['10.144.12.2']
gateway_ip=10.144.12.1 for client client_0


Finished 1 tasks (create_macvlan_namespaces) on {'gros-83.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 1 namespaces for client_0 (with gateway 10.144.12.1)


### Setting up GRE tunnels between routers in different clusters

This step will create GRE tunnels between each pair of routers as defined in the topology file. The endpoints of the tunnels use the production IP of the nodes.

In [4]:
experiment.setup_gre_tunnels()

Output()

Link 0: gre1(router_server, 192.168.0.1) <-> gre1(router_client_0, 192.168.0.2)


Finished 1 tasks (setup_gre_router_server) on {'parasilo-18.rennes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Created 1 GRE tunnels on router_server (parasilo-18.rennes.grid5000.fr)


Finished 1 tasks (setup_gre_router_client_0) on {'gros-82.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Created 1 GRE tunnels on router_client_0 (gros-82.nancy.grid5000.fr)
Router tunnels: {'router_server': [{'iface': 'gre1', 'ip': '192.168.0.1', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}], 'router_client_0': [{'iface': 'gre1', 'ip': '192.168.0.2', 'network': '192.168.0.0', 'tunnel_subnet': IPv4Network('192.168.0.0/30')}]}


#### FRRouting setup

With GRE tunnels setup between rotuers, we can now configure and start FRRouting. The frr configuration template defined in the topology file will be used as a base.

In [5]:
experiment.frrouting_setup()
experiment.setup_default_routes()

Output()

Finished 1 tasks (restart_frr_parasilo-18.rennes.grid5000.fr) on 
{'parasilo-18.rennes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_server] parasilo-18.rennes.grid5000.fr  prod=10.158.8.1  loopback=10.158.11.254  gateway=172.16.111.254  tunnels=1


Output()

Finished 1 tasks (restart_frr_gros-82.nancy.grid5000.fr) on {'gros-82.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

[router_client_0] gros-82.nancy.grid5000.fr  prod=10.144.12.1  loopback=10.144.15.254  gateway=172.16.79.254  tunnels=1
172.16.97.18
172.16.66.82
172.16.66.82
setting default routes on 3 nodes
default via 172.16.79.254 dev eno1
gros-83.nancy.grid5000.fr's default route is 172.16.66.82 on eno1
default via 172.16.79.254 dev eno1
gros-87.nancy.grid5000.fr's default route is 172.16.66.82 on eno1
default via 172.16.111.254 dev eno1
parasilo-26.rennes.grid5000.fr's default route is 172.16.97.18 on eno1


### Upload binary files over to nodes

We build the executables locally first

In [6]:
!cd /home/corentin/fcquic_applications_master_thesis/fcquic_relay && cargo build --release

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:474:22
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets> {
    |                      ^^^^^^^^^                        ^^^^^^ the same lifetime is hidden here
    |                      |
    |                      the lifetime is elided here
    |
    = help: the same lifetime is referred to in inconsistent ways, making the signature confusing
    = note: `#[warn(mismatched_lifetime_syntaxes)]` on by default
help: use `'_` for type paths
    |
474 |     pub fn get_bytes(&mut self, len: usize) -> Result<Octets<'_>> {
    |                                                             ++++

   --> /home/corentin/fcquic_applications_master_thesis/multicast-quic/octets/src/lib.rs:491:26
    |
491 |     pub fn get_bytes_mut(&mut self, len: usize) -> Result<OctetsMut> {
    |                          ^^^^^^^^^                        ^^^^^^^^^ the same lifetime is hidden here
    | 

Then we push them to the nodes

In [7]:
import enoslib as en

experiment.push_binaries(
    # TODO: change these paths with the path to your binaries and certificates
    bin_dir="/home/corentin/fcquic_applications_master_thesis/fcquic_relay/target/release",
    cert_dir="/home/corentin/fcquic_applications_master_thesis/fcquic_relay",
)

res = en.run_command(
    "sysctl -w net.core.rmem_default=26214400 && sysctl -w net.core.rmem_max=26214400",
    roles=experiment.roles,
)
print("errors: " + str([out.stderr for out in res.filter(status=en.STATUS_FAILED)]))

Output()

Finished 2 tasks (file,copy) on {'gros-83.nancy.grid5000.fr', 
'parasilo-26.rennes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Pushed ['server', 'client'] to 2 server/client node(s)


...

Output()

Pushed ['fcquic_relay', 'app_relay'] to 1 relay node(s)


Finished 1 tasks (sysctl -w net.core.rmem_default=26214400 && sysctl -w 
net.core.rmem_max=26214400) on {'gros-83.nancy.grid5000.fr', 
'parasilo-26.rennes.grid5000.fr', 'gros-87.nancy.grid5000.fr', 
'parasilo-18.rennes.grid5000.fr', 'gros-82.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

errors: []


### Running the relay experiment

Experiment.py provides some basic blocks that should (ideally) allow you to define your own custom experiments.
Below you will find the code for the evaluation of two types of Flexicast QUIC relays, this should hopefully provide enough information.


In [8]:
import concurrent.futures
from dataclasses import dataclass
from datetime import datetime, timedelta
from typing import Literal
import time

import enoslib as en

from g5k_eval.cpuload import collect_cpuload, install_cpuload
from g5k_eval.experiment import (
    EvalConfig,
    MetricSpec,
    collect_results,
    run_eval,
)
from g5k_eval.remote import (
    run_cmd_bg_enos,
    run_cmd_ssh_parallel,
    send_pkill_hosts,
    ssh_bg,
)

RelayVersion = Literal["none", "RELAY", "APP_RELAY"]


@dataclass
class RunConfig:
    test_index: int  # 0 for latency test, 1 for segmentation test
    relay_version: RelayVersion
    additional_data_size: int
    lambda_: float
    test_length: int


@dataclass
class RelayEvalConfig(EvalConfig):
    ready_sleep_relay: int = 1
    ready_sleep_clients: int = 2
    post_test_buffer: int = 3
    bin_log_level: str = "info"
    cert_path: str = "/tmp"
    server_bin: str = "/tmp/bin/server"
    client_bin: str = "/tmp/bin/client"
    fcquic_relay_bin: str = "/tmp/bin/fcquic_relay"
    app_relay_bin: str = "/tmp/bin/app_relay"
    remote_log_root: str = "/tmp/logs"
    num_ns_per_client: int = experiment.topology.netns_per_client
    cc_algo: str = "disabled"
    fallback_delay: int = 10000
    interval: int = 100
    poisson: bool = True
    use_system_time: bool = True
    server_cpus: str = "0-1"  # taskset -c range for server


# define the results to extract from the client logs, one csv output file is emitted for each metric
METRICS = [
    MetricSpec(key="LATENCY", column="y_LATENCY"),
    #  you can add more result types here, e.g.:
    # MetricSpec(key="THROUGHPUT", column="y_THROUGHPUT"),
]

We can now define specific tests based on our `RunConfig`.

In the case of the relay evaluation, we wanted to perform a test to measure the latency overhead of the relay versions with packets of constant size, but we also wanted to perform another test to measure the impact of QUIC packet segmentation on the latency with relays.

In [9]:
def latency_matrix():
    return [
        RunConfig(0, relay_version, 1100, lambda_=50000, test_length=10)
        for relay_version in ("none", "RELAY", "APP_RELAY")
    ]


def segmentation_matrix():
    return [
        RunConfig(1, relay_version, size, lambda_=20000, test_length=10)
        for relay_version in ("none", "RELAY", "APP_RELAY")
        for size in (1100, 2200)
    ]

Now that the experiment is defined, we need to specify the commands that will be ran on the nodes.

Here, we define the command to run the server, the relays, and the clients

In [10]:
# ---------- command builders ----------
def server_cmd(cfg, rc, server_ip, run_dir):
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/server"
    return (
        f"mkdir -p {qlog} && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} taskset -c {cfg.server_cpus} {cfg.server_bin} "
        f"--cert-path {cfg.cert_path} --src {server_ip}:4433 --mc-src-addr {server_ip}:4443 "
        f"--test-mode --flexicast --fc-timer 0 --fall-back-delay {cfg.fallback_delay} "
        f"--unicast --fec-scheduler noredundancy --length {length} "
        f"--cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo}"
    )


def relay_cmd(cfg, rc, server_ip, run_dir):
    bin_ = cfg.fcquic_relay_bin if rc.relay_version == "RELAY" else cfg.app_relay_bin
    length = rc.test_length * 2
    qlog = f"{run_dir}/qlog/relay"
    return (
        f"mkdir -p {qlog} && "
        f"CURRENT_RELAY_IP=$(ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1) && "
        f"env QLOGDIR={qlog} RUST_LOG_STYLE=never RUST_BACKTRACE=full "
        f"RUST_LOG={cfg.bin_log_level} {bin_} "
        f"https://{server_ip}:4433 --src $CURRENT_RELAY_IP:4433 --mc-src-addr $CURRENT_RELAY_IP:4443 "
        f"--cert-path {cfg.cert_path} --test-mode --flexicast --length {length} "
        f"--fc-timer 0 --fall-back-delay {cfg.fallback_delay} --unicast "
        f"--fec-scheduler noredundancy --cc-algorithm {cfg.cc_algo} --fc-cwnd {cfg.cc_algo}"
    )


# IMPORTANT NOTE: since we have multiple network namespaces defined on each client machine,
# we can run processes in these namespaces using the naming scheme "client-$NS_IDX" (with NS_IDX going from the number of 0 to NSs)
def client_loop_cmd(cfg, rc, server_ip, relay_ips, run_dir, node_id, sleep_deadline_ts):
    relay_args = (
        " ".join(f"--relay-ips={ip}" for ip in relay_ips)
        if rc.relay_version != "none"
        else ""
    )
    poisson = f"--poisson --lambda {rc.lambda_}" if cfg.poisson else ""
    systime = "--use-system-time" if cfg.use_system_time else ""
    return f"""
mkdir -p {run_dir}/client
pids=()
for NS_IDX in $(seq $(( {cfg.num_ns_per_client} - 1 )) -1 0); do
    GLOBAL_IDX=$(( {node_id} * {cfg.num_ns_per_client} + NS_IDX ))
    CLIENT_ID=$(( GLOBAL_IDX + 1 ))
    NS_NAME="client-$NS_IDX"
    CLIENT_IP=$(ip netns exec $NS_NAME ip -f inet addr show | grep inet | tail -1 | awk '{{print $2}}' | cut -d'/' -f1)
    EXTRA=""; [ "$CLIENT_ID" = "1" ] && EXTRA="--sender"
    ip netns exec $NS_NAME env RUST_LOG_STYLE=never RUST_BACKTRACE=full RUST_LOG={cfg.bin_log_level} \\
        {cfg.client_bin} --server-ip {server_ip} --port 4433 {relay_args} \\
        -l $CLIENT_IP --flexicast -u CLIENT$CLIENT_ID --length {rc.test_length} --test-mode \\
        --conn-sleep-length 1 --test-start-ts {sleep_deadline_ts} --interval {cfg.interval} \\
        --additional-data-size {rc.additional_data_size} --cc-algorithm {cfg.cc_algo} \\
        --show-own-messages {poisson} {systime} $EXTRA \\
        > {run_dir}/client/client_$CLIENT_ID.stdout \\
        2> {run_dir}/client/client_$CLIENT_ID.stderr < /dev/null < /dev/null &
    pids+=($!)
done
for pid in "${{pids[@]}}"; do wait $pid; done
"""


# ---------- one run of the relay experiment ----------
def run_once(cfg, rc, run_index, test_name):
    """Run one iteration: start the server (and relays), start the clients in
    their namespaces, wait for the test to finish, then collect the results."""
    roles_dict = experiment.roles
    node_ips = experiment.node_ips
    relay_ips = [
        node_ips[f"relay_{i}"][0]
        for i in range(len(experiment.topology.client_clusters))
    ]
    server_ip = node_ips["server"][0]
    run_id = f"run_t{rc.test_index}_{rc.relay_version}_sz{rc.additional_data_size}_r{run_index}"
    run_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}"

    relay_hosts = [
        h
        for i in range(len(experiment.topology.client_clusters))
        for h in roles_dict[f"relay_{i}"]
    ]

    # make sure that each client is root because it has to start the clients in network namespaces
    client_hosts = [
        en.Host(h.address, alias=h.alias, user="root", extra=h.extra)
        for h in roles_dict["client"]
    ]
    all_hosts = roles_dict["server"] + client_hosts + relay_hosts

    # create the dirs on all of the hosts
    run_cmd_ssh_parallel(
        f"mkdir -p {run_dir}/server {run_dir}/relay {run_dir}/client {run_dir}/qlog",
        all_hosts,
    )

    # make sure all programs are stopped
    send_pkill_hosts(all_hosts, ["server", "fcquic_relay", "app_relay", "client"])
    time.sleep(1)

    # start server in bg
    run_cmd_bg_enos(
        server_cmd(cfg, rc, server_ip, run_dir),
        roles_dict["server"],
        stdout=f"{run_dir}/server/server.stdout",
        stderr=f"{run_dir}/server/server.stderr",
        task_name="server",
    )

    # start CPU load monitor (server only)
    if cfg.monitor_cpu:
        cpuload_py = f"{run_dir}/server/cpuload.py"
        cpuload_csv = f"{run_dir}/server/cpuload.csv"
        install_cpuload(roles_dict["server"], cpuload_py)

        run_cmd_bg_enos(
            f"python3 -u {cpuload_py} {cpuload_csv} {rc.test_length + 5} 0 {cfg.cpu_max}",
            roles_dict["server"],
            stdout=f"{run_dir}/server/cpuload.stdout",
            stderr=f"{run_dir}/server/cpuload.stderr",
            task_name="start_cpuload",
        )

    time.sleep(cfg.ready_sleep_relay)

    # start relays if we're testing with them
    if rc.relay_version != "none":
        run_cmd_bg_enos(
            relay_cmd(cfg, rc, server_ip, run_dir),
            relay_hosts,
            stdout=f"{run_dir}/relay/relay_$(hostname).stdout",
            stderr=f"{run_dir}/relay/relay_$(hostname).stderr",
            task_name="relay",
        )

    # pick a timestamp in 5 seconds, we pass this to all of the clients that will all wait until that timestamp is reached before starting
    datetime_now = datetime.now()
    sleep_deadline = datetime_now + timedelta(seconds=5)
    sleep_deadline_ts = sleep_deadline.timestamp()

    # start all clients in // to make them start kinda at the same time
    def _start_client(node_id, h):
        cmd = client_loop_cmd(
            cfg, rc, server_ip, relay_ips, run_dir, node_id, sleep_deadline_ts
        )
        ssh_bg(
            cmd,
            h,
            stdout=f"{run_dir}/client/loop_{node_id}.stdout",
            stderr=f"{run_dir}/client/loop_{node_id}.stderr",
        )

    with concurrent.futures.ThreadPoolExecutor(
        max_workers=max(1, len(client_hosts))
    ) as ex:
        list(ex.map(lambda p: _start_client(*p), list(enumerate(client_hosts))))

    # wait for test duration to pass
    time.sleep(rc.test_length + cfg.post_test_buffer)

    send_pkill_hosts(all_hosts, ["server", "fcquic_relay", "app_relay", "client"])

    time.sleep(1)

    results = collect_results(cfg, client_hosts, run_dir, test_name, METRICS)
    cpu_samples = (
        collect_cpuload(cfg, roles_dict["server"][0], run_dir, test_name)
        if cfg.monitor_cpu
        else []
    )
    return results, cpu_samples

Lauching the test

In [11]:
LATENCY_TEST = True
N_RUNS = 1

cfg = RelayEvalConfig(n_runs=N_RUNS, cpu_max=2)
relay_ips = [
    experiment.node_ips[f"relay_{i}"][0]
    for i in range(len(experiment.topology.client_clusters))
]
now = datetime.now().strftime("%d-%m-%H-%M%p")


test_prefix = "serv_2thr_"
test_name = ""
matrix = None

if LATENCY_TEST:
    test_name = f"{test_prefix}latency_test_large_relay_topo_{now}"
    matrix = latency_matrix()
else:
    test_name = f"{test_prefix}segmentation_test_large_relay_topo_{now}"
    matrix = segmentation_matrix()


def relay_row_fields(rc):
    # columns identifying each run in the result CSVs
    return {
        "test_index": rc.test_index,
        "ADDITIONAL_DATA_SIZE": rc.additional_data_size,
        "RELAY_VERSION": f'"{rc.relay_version}"',
    }


run_eval(
    matrix,
    cfg,
    run_once=run_once,
    test_name=test_name,
    row_fields=relay_row_fields,
    metrics=METRICS,
)

=> {'test_index': 0, 'ADDITIONAL_DATA_SIZE': 1100, 'RELAY_VERSION': '"none"'} run=0 (attempt 1)


Output()

Finished 1 tasks (server) on {'parasilo-26.rennes.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

...

Output()

Finished 1 tasks (relay) on {'gros-87.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

-> collected {'LATENCY': 873} samples, 42 cpu samples


Test finished in 77.21081471443176 seconds
Writing CSV file...
wrote npf-out/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM.csv (2619 rows)
Writing CSV file...
wrote npf-out/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM_cpu.csv (126 rows)


{'LATENCY': PosixPath('npf-out/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM.csv')}

#### Downloading SQLOGs from server and relay

In [13]:
import subprocess
from pathlib import Path

# uses cfg, matrix, test_name and N_RUNS from the launching cell above
local_base = Path(f"./sqlogs/{test_name}")

for run_conf in matrix:
    for run_index in range(N_RUNS):
        run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

        remote_qlog_dir = f"{cfg.remote_log_root}/{test_name}/{run_id}/qlog/server"
        local_dir = local_base / run_id / "server"
        local_dir.mkdir(parents=True, exist_ok=True)

        # download the sqlogs from the server only
        for node in experiment.roles["server"]:
            host = node.address
            print(f"downloading sqlogs from {host} to {remote_qlog_dir}")
            subprocess.run(
                [
                    "rsync",
                    "-az",
                    # "-o LogLevel=ERROR",
                    "--include=*.sqlog",
                    "--exclude=*",
                    f"root@{host}:{remote_qlog_dir}/",
                    f"{local_dir}/",
                ],
                check=False,
            )

print(f"results: {local_base}")

downloading sqlogs from parasilo-26.rennes.grid5000.fr to /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/run_t0_none_sz1100_r0/qlog/server


downloading sqlogs from parasilo-26.rennes.grid5000.fr to /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/run_t0_RELAY_sz1100_r0/qlog/server


downloading sqlogs from parasilo-26.rennes.grid5000.fr to /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/run_t0_APP_RELAY_sz1100_r0/qlog/server


results: sqlogs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM


#### Merging SQLOG files together


In [14]:
from pathlib import Path
import sys
import tempfile
import re
import json
import csv


def merge_sqlogs(files, output):
    control_chars = re.compile(r"[\x00-\x08\x0b-\x1f\x7f]")

    with open(output, "w") as out:
        for i, f in enumerate(files):
            with open(f) as src:
                first = True
                for j, line in enumerate(src):
                    if j == 0 and i > 0:
                        # if i > 0, then we wrote the header once already, so now skip the headers (first lines of sqlog files: j==0)
                        continue

                    line = line.rstrip() + "\n"
                    line = control_chars.sub("", line)
                    out.write(control_chars.sub("", line))


def extract_path_acks(sqlog, csv_out):

    # we need to get the path_ack lengths from the sqlogs
    # go through the merged sqlog, and append to a csv file the time and length of each path_ack we see
    with open(sqlog) as src, open(csv_out, "w", newline="") as out:
        writer = csv.writer(out)
        writer.writerow(["time", "length"])

        for line in src:
            line = line.strip()
            if not line:
                continue

            try:
                event = json.loads(line)
            except json.JSONDecodeError:
                # idk why so many log entries are broken
                continue

            # e.g.
            # {"time":12.632589,"name":"transport:packet_received","data":{"header":{"packet_type":"1RTT","packet_number":4},"raw":{"length":1155,"payload_length":1138},"frames":[{"frame_type":"path_ack","path_identifier":0,"ack_delay":0.085,"acked_ranges":[[3,3]]},{"frame_type":"path_new_connection_id","path_id":1,"sequence_number":0,"retire_prior_to":0,"connection_id_length":16,"connection_id":"ab44f5dde5157072f203e53c8d76836d","stateless_reset_token":"536abfd4dc2107e6e1188cbba08f4ce1"},{"frame_type":"padding","payload_length":1071}]}}
            if event.get("name") == "transport:packet_received":
                frames = event.get("data", {}).get("frames", []) or []

                for frame in frames:
                    if frame.get("frame_type") == "path_ack":
                        # if we do have a path_ack frame (migth contain other stuff), get the length
                        length = event.get("data", {}).get("raw", {}).get("length")

                        if length is not None:
                            writer.writerow([event.get("time"), length])


local_base = Path(f"./sqlogs/{test_name}")

for relay_test in ["none", "RELAY", "APP_RELAY"]:

    trace_files = []
    for run_conf in matrix:
        if run_conf.relay_version != relay_test:
            continue

        for run_index in range(N_RUNS):
            run_id = f"run_t{run_conf.test_index}_{run_conf.relay_version}_sz{run_conf.additional_data_size}_r{run_index}"

            server_dir = local_base / run_id / "server"

            for file in sorted(server_dir.glob("server-server-*.sqlog")):
                trace_files.append(file)

    if not trace_files:
        print(f"no files found for {relay_test}")
        continue

    print(f"relay={relay_test}: {len(trace_files)} trace files")

    temp_dir = Path(tempfile.mkdtemp(prefix="ackrate_"))
    merged_log = temp_dir / f"merged_{relay_test}.sqlog"

    merge_sqlogs(trace_files, merged_log)

    merged_csv = Path(f"./npf-out/ack_rate_{test_name}") / f"{relay_test}.csv"
    merged_csv.parent.mkdir(parents=True, exist_ok=True)

    extract_path_acks(merged_log, merged_csv)
    print(f"path_ack csv: {merged_csv}")

relay=none: 1 trace files
path_ack csv: npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/none.csv
relay=RELAY: 1 trace files
path_ack csv: npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/RELAY.csv
relay=APP_RELAY: 1 trace files
path_ack csv: npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/APP_RELAY.csv


### Graphing the results


In [16]:
import subprocess
from pathlib import Path

INSET_GRAPHS = True
NO_TITLE = True
out_path = f"./graphs/{test_name}/"
output_path = Path(out_path)
output_path.mkdir(parents=True, exist_ok=True)
subprocess.run(
    [
        "./relay_graphs.py",
        f"./npf-out/{test_name}.csv",  # input csv paht
        out_path,  # out path
        test_name,
        f"./npf-out/ack_rate_{test_name}/",  # ack_rate_path
        f"./npf-out/{test_name}_cpu.csv",  # cpu_csv_path
        *(["--inset"] if INSET_GRAPHS else []),
        *(
            ["--no-title"] if NO_TITLE else []
        ),  # list unpacking, this avoids the empty ""
    ],
    check=True,
)

ADDITIONAL_DATA_SIZE values: [np.int64(1100)]
No relay samples: 873
FCQUIC relay samples: 873
APP relay samples: 873
Per run breakdown
none:
  Run 0: 873 samples
RELAY:
  Run 0: 873 samples
APP_RELAY:
  Run 0: 873 samples
min length of the dataframes: 873
only one additional data size, skipping mean/median plot.


CompletedProcess(args=['./relay_graphs.py', './npf-out/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM.csv', './graphs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/', 'serv_2thr_latency_test_large_relay_topo_14-09-14-41PM', './npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/', './npf-out/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM_cpu.csv', '--inset', '--no-title'], returncode=0)

#### Compressing the csv results

In [17]:
import subprocess

# compress all related files in one tarball
subprocess.run(
    [
        "tar",
        "czf",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

# move archive to the graph dir of the test
subprocess.run(
    [
        "mv",
        f"./npf-out/all_{test_name}.tar.gz",
        f"./graphs/{test_name}/{test_name}.tar.gz",
    ],
    check=True,
)

# delete the csvs and directories
subprocess.run(
    [
        "rm",
        f"./npf-out/{test_name}.csv",
        f"./npf-out/{test_name}_cpu.csv",
    ],
    check=True,
)
subprocess.run(
    [
        "rm",
        "-rf",
        f"./npf-out/ack_rate_{test_name}/",
        f"./npf-out/raw/{test_name}/",
        f"./sqlogs/{test_name}/",
    ],
    check=True,
)

CompletedProcess(args=['rm', '-rf', './npf-out/ack_rate_serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/', './npf-out/raw/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/', './sqlogs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/'], returncode=0)

Opposite code to unarchive the results, in order to regenerate graphs if needed

In [ ]:
import subprocess
from pathlib import Path

# og_name = "sserv_2thr_latency_test_large_relay_topo_26-04-21-39PM_481mbps"
test_name = "serv_2thr_latency_test_large_relay_topo_26-04-21-39PM"

archive_path = Path(f"./graphs/{test_name}/{test_name}.tar.gz")

if archive_path.exists():
    print(f"decompressing {archive_path}...")

    Path("./npf-out/").mkdir(parents=True, exist_ok=True)

    subprocess.run(
        [
            "tar",
            "xzf",
            str(archive_path),
            "-C",
            "./",
        ],
        check=True,
    )
    print(f"decompressed files to ./npf-out/ and ./sqlogs/")
else:
    print(f"Couldn't find: {archive_path}")

#### Deleting log files from all clusters

In [18]:
import subprocess
from pathlib import Path

remote_log_root = "/tmp/logs"

if LATENCY_TEST:
    matrix = latency_matrix()
else:
    matrix = segmentation_matrix()

for run_conf in matrix:

    remote_qlog_dir = f"{remote_log_root}/{test_name}/"

    en.run_command(
        f"rm -rf {remote_qlog_dir}",
        roles=experiment.roles["client"]
        + experiment.roles["relay"]
        + experiment.roles["server"],
    )

print("done deleting sqlog files")

Output()

Finished 1 tasks (rm -rf /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/) on
{'gros-83.nancy.grid5000.fr', 'parasilo-26.rennes.grid5000.fr', 'gros-87.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Finished 1 tasks (rm -rf /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/) on
{'gros-83.nancy.grid5000.fr', 'parasilo-26.rennes.grid5000.fr', 'gros-87.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

Output()

Finished 1 tasks (rm -rf /tmp/logs/serv_2thr_latency_test_large_relay_topo_14-09-14-41PM/) on
{'gros-83.nancy.grid5000.fr', 'parasilo-26.rennes.grid5000.fr', 'gros-87.nancy.grid5000.fr'}

─────────────────────────────────────────────────────────────────────────────────────────────

done deleting sqlog files


## Important: Stopping the current booking
Always, always stop your booking if you are done earlier.

In [17]:
experiment.stop_reservation()

INFO     [G5k] Reloading 2205568 from lille                              ]8;id=385168;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=411305;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Reloading 6926385 from nancy                              ]8;id=94577;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=73268;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#169\169]8;;\

INFO     [G5k] Killing the job (lille, 2205568)                          ]8;id=419023;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=114948;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#278\278]8;;\

INFO     [G5k] Job killed (lille, 2205568)                               ]8;id=686593;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=68762;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#259\259]8;;\

INFO     [G5k] Killing the job (nancy, 6926385)                          ]8;id=39567;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=289974;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#278\278]8;;\

INFO     [G5k] Job killed (nancy, 6926385)                               ]8;id=703042;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py\g5k_api_utils.py]8;;\:]8;id=867437;file:///home/corentin/.local/lib/python3.12/site-packages/enoslib/infra/enos_g5k/g5k_api_utils.py#259\259]8;;\

Reservation stopped.
